# 工具参数验证

`nova_agent` 内置基于 `jsonschema` 的参数验证，也允许全局禁用或清空缓存。

本 Notebook 不依赖 LLM，可离线运行。


In [1]:
from nova_ai import Tool, ToolCall
from nova_agent import (
    validate_tool_call,
    validate_tool_arguments,
    set_validation_enabled,
    clear_validator_cache,
)

weather_tool = Tool(
    name="get_weather",
    description="查询天气",
    parameters={
        "type": "object",
        "properties": {
            "city": {"type": "string"},
            "unit": {"type": "string", "enum": ["C", "F"]},
        },
        "required": ["city"],
    },
)

# 合法调用
valid_call = ToolCall(id="1", name="get_weather", arguments={"city": "杭州", "unit": "C"})
print("合法:", validate_tool_arguments(weather_tool, valid_call))

# 非法调用：缺少必填参数
try:
    invalid_call = ToolCall(id="2", name="get_weather", arguments={"unit": "C"})
    validate_tool_arguments(weather_tool, invalid_call)
except ValueError as e:
    print("\n验证失败:\n", e)


合法: {'city': '杭州', 'unit': 'C'}

验证失败:
 Validation failed for tool "get_weather":
  - city: is required

Received arguments:
{
  "unit": "C"
}


In [2]:
# 通过名称查找工具并验证
tools = [weather_tool]
print(validate_tool_call(tools, valid_call))

# 禁用验证
set_validation_enabled(False)
print("禁用验证后:", validate_tool_arguments(weather_tool, invalid_call))

# 重新启用
set_validation_enabled(True)
clear_validator_cache()


{'city': '杭州', 'unit': 'C'}
禁用验证后: {'unit': 'C'}
